# Air Quality SLM — continue training on a Colab GPU

This notebook runs **`src/continue_training.py`** from your existing project, unchanged, on a free
Colab GPU. It continues from your trained `pytorch_model.bin` (never from random weights).

### What to upload
Zip the following from `C:\Users\sanja\OneDrive\Documents\JAIN_Academics\SEM3\DT\SLM` (≈100 MB total):

```
src/                     (train_slm.py, continue_training.py, evaluate_finetuned.py,
                          export_gguf_from_checkpoint.py, prepare_training_data.py)
models/air_quality_slm/  (pytorch_model.bin, config.json, tokenizer.json,
                          tokenizer_config.json, chat_template.jinja)
training/data_bin/       (train.bin, val.bin, test.bin)
data/training/           (val_instructions.jsonl, train_instructions.jsonl)
docs/                    (optional, for the log file)
```

`data/processed/*.csv` is only needed if you want to rebuild the `.bin` files — it is not needed here.

### Why GPU
On the local machine (8 GB RAM, no CUDA) training runs at ~720 tokens/s; one full epoch of the
3.04 M-token corpus takes about 70 minutes. A T4 does the same epoch in a few minutes.

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# Upload your project zip (SLM.zip)
from google.colab import files
up = files.upload()
print('uploaded:', list(up))

In [ ]:
import os, zipfile, glob, shutil

os.makedirs('/content/SLM', exist_ok=True)
zip_name = [f for f in glob.glob('/content/*.zip')][0]
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/SLM')

# handle a zip that contains a single top-level folder
if not os.path.exists('/content/SLM/src'):
    inner = glob.glob('/content/SLM/*/')
    if inner:
        for item in os.listdir(inner[0]):
            shutil.move(os.path.join(inner[0], item), '/content/SLM/')
        os.rmdir(inner[0])

%cd /content/SLM
!pip -q install gguf pandas tqdm psutil
!ls
!ls training/data_bin models/air_quality_slm

## Continue training

`--steps 750` ≈ one full epoch at 4096 tokens/step (3.04 M tokens). Validation runs every 50 steps
and the best checkpoint is written to `models/air_quality_slm_finetuned/best_model.pt`.

Early stopping is on: if validation loss stops improving for 6 consecutive evaluations it stops.

In [ ]:
!python -u src/continue_training.py \
  --device cuda \
  --steps 750 \
  --block 128 \
  --batch 16 \
  --grad-accum 2 \
  --lr 2e-4 \
  --eval-every 50 \
  --patience 6 \
  --threads 2

## Promote the best checkpoint and compare

`continue_training.py` writes `best_model.pt` (a dict). This cell extracts its `model_state_dict`
to `pytorch_model.bin` and copies the config with the new 128-token context, then runs the
before/after generation comparison.

In [ ]:
import json, shutil, torch
from pathlib import Path

out = Path('models/air_quality_slm_finetuned')
ck = torch.load(out / 'best_model.pt', map_location='cpu')
torch.save(ck['model_state_dict'], out / 'pytorch_model.bin')
print('saved best step', ck['step'], 'val_loss', round(ck['val_loss'], 5))

cfg = json.loads(Path('models/air_quality_slm/config.json').read_text())
cfg['max_position_embeddings'] = ck.get('block', 128)
Path(out / 'config.json').write_text(json.dumps(cfg, indent=2))
for f in ['tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja']:
    shutil.copy(f'models/air_quality_slm/{f}', out / f)

!python -u src/evaluate_finetuned.py --rows 2 --out docs/comparison_finetune.json

In [ ]:
# Download the improved checkpoint and the comparison report
from google.colab import files
files.download('models/air_quality_slm_finetuned/pytorch_model.bin')
files.download('docs/comparison_finetune.json')
files.download('docs/training_log_finetune.jsonl')

## Export to GGUF (run back on your machine)

The GGUF export needs the tokenizer directory; it is simplest to run locally after copying
`pytorch_model.bin` back:

```powershell
python src/export_gguf_from_checkpoint.py `
  --checkpoint models/air_quality_slm_finetuned/pytorch_model.bin `
  --config     models/air_quality_slm_finetuned/config.json `
  --tokenizer  models/air_quality_slm `
  --output     models/air_quality_slm_finetuned/air_quality_slm_finetuned.gguf `
  --context 128
```

### Honest expectations
More training makes this model a better **air-quality record formatter**: it should produce
complete, correctly-structured reports and answers with the right field names and units. It will
not become a general chatbot, and it will not reliably recall the true sensor reading for an
arbitrary date — there is no retrieval, and the training signal is 150k datapoints of template text.